In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [3]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [4]:
benchmarks = [
    # High-sharing Apps
    'gemm',
    'lu',
    'spmv',
    'mm3',
    'mm2',
    'matrixmultiplication',
    'syr2k',
    'gups',    
    'mis',
    'matrixtranspose',
    'stencil2d',
    'gaussian',

    # Low-sharing Apps
    'fdtd2d',
    'fastwalshtransform',
    'jacobi1d',
    'relu',
    'pagerank',
    'bfs',
    'color',
    'convolution2d',
    'sssp',
    'fir',
    'fft',
    'bitonicsort',
]

In [5]:
num_chiplets = 36
ns = 0.000000001

In [6]:
date = "20260221-223953"
base_dir=f'{username}/mgvm/results/memory-traces'
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"

In [7]:
chip_cols = [str(i) for i in range(num_chiplets)]

In [ ]:
results = []

for benchmark in benchmarks:
    trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

    if not os.path.exists(trace_file):
        print(f"Skip: No {trace_file}")
        continue

    print(f"Processing {benchmark}...")
    df = pd.read_csv(trace_file, skipinitialspace=True)

    # ===== addr별 집계 =====
    agg_dict = {col: 'sum' for col in chip_cols}
    agg_dict['rw'] = 'max'
    agg_dict['access_count'] = 'sum'
    agg_dict['owner'] = 'first'
    grouped = df.groupby('addr').agg(agg_dict).reset_index()

    # 1. 0~35 컬럼명만 안전하게 추출
    valid_chip_strs = {str(i) for i in range(36)}
    chip_cols = [col for col in grouped.columns if str(col) in valid_chip_strs]

    # 2. owner 값을 "순수 NumPy 문자열 배열"로 변환 후 2차원(N, 1)으로 세움
    owner_vals = np.array(grouped['owner'], dtype=str)[:, None]  # Shape: (307172, 1)

    # 3. 컬럼명 배열도 "순수 NumPy 문자열 배열"로 변환
    col_vals = np.array(chip_cols, dtype=str)  # Shape: (36,)

    # 4. 브로드캐스팅 비교 (Pandas 간섭 없이 순수 NumPy C엔진으로 초고속 연산됨)
    is_owner = owner_vals == col_vals  # Shape: (307172, 36) 의 True/False 마스크 생성

    # 5. 마스크 적용하여 값 0으로 치환
    grouped[chip_cols] = grouped[chip_cols].mask(is_owner, 0)

    is_shared_mask = (grouped[chip_cols] != 0).any(axis=1)
    shared_group = grouped[is_shared_mask]
    private_group = grouped[~is_shared_mask]  # Not 연산자로 간단하게 추출


    

In [15]:
benchmark = 'bfs'

trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"

df = pd.read_csv(trace_file, skipinitialspace=True)

# ===== addr별 집계 =====
agg_dict = {col: 'sum' for col in chip_cols}
agg_dict['rw'] = 'max'
agg_dict['access_count'] = 'sum'
agg_dict['owner'] = 'first'
grouped = df.groupby('addr').agg(agg_dict).reset_index()

# 1. 0~35 컬럼명만 안전하게 추출
valid_chip_strs = {str(i) for i in range(36)}
chip_cols = [col for col in grouped.columns if str(col) in valid_chip_strs]

# 2. owner 값을 "순수 NumPy 문자열 배열"로 변환 후 2차원(N, 1)으로 세움
owner_vals = np.array(grouped['owner'], dtype=str)[:, None]  # Shape: (307172, 1)

# 3. 컬럼명 배열도 "순수 NumPy 문자열 배열"로 변환
col_vals = np.array(chip_cols, dtype=str)  # Shape: (36,)

# 4. 브로드캐스팅 비교 (Pandas 간섭 없이 순수 NumPy C엔진으로 초고속 연산됨)
is_owner = owner_vals == col_vals  # Shape: (307172, 36) 의 True/False 마스크 생성

# 5. 마스크 적용하여 값 0으로 치환
grouped[chip_cols] = grouped[chip_cols].mask(is_owner, 0)

is_shared_mask = (grouped[chip_cols] != 0).any(axis=1)
shared_group = grouped[is_shared_mask]
private_group = grouped[~is_shared_mask]  # Not 연산자로 간단하게 추출

In [23]:
def plot_addr_hotness_timeseries(df, benchmark, time_interval=10000, top_n=10,
                                  out_base="./addr-hotness"):
    """
    X축: interval
    Y축: access_count (per interval 합계)
    색상: 주소별 구분
    → 상위 주소가 모든 시간대에 걸쳐 일정하게 찍히면 "persistent hot data" 확인됨
    """
    df = df.copy()
    t_min = df['time'].min()
    df['interval'] = ((df['time'] - t_min) // time_interval).astype(int)

    agg = (df.groupby(['interval', 'addr'])['access_count']
             .sum()
             .reset_index(name='total_access'))

    top_addrs = (agg.groupby('addr')['total_access']
                    .sum()
                    .nlargest(top_n)
                    .index.tolist())
    agg_top = agg[agg['addr'].isin(top_addrs)].copy()

    palette = sns.color_palette("tab10", n_colors=top_n)
    addr_color = {a: palette[i] for i, a in enumerate(top_addrs)}

    fig, ax = plt.subplots(figsize=(14, 5))

    for addr in top_addrs:
        sub = agg_top[agg_top['addr'] == addr]
        ax.scatter(sub['interval'], sub['total_access'],
                   s=18, alpha=0.75,
                   color=addr_color[addr],
                   label=f'0x{int(addr):X}')

    ax.set_title(
        f'[{benchmark}]  Top-{top_n} Hot Addresses — Access Count per Interval',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('Interval  (time unit: {:,})'.format(time_interval), fontsize=11)
    ax.set_ylabel('Access Count (sum per interval)', fontsize=11)
    ax.legend(title='Address (hex)', bbox_to_anchor=(1.01, 1),
              loc='upper left', fontsize=8, frameon=True)

    sns.despine()
    plt.tight_layout()
    out_dir = f"{out_base}/{benchmark}"
    os.makedirs(out_dir, exist_ok=True)
    out_file = f"{out_dir}/{benchmark}_addr_access_timeseries.png"
    plt.savefig(out_file, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"✅ Timeseries saved: {out_file}")


# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────
benchmark = 'syr2k'

trace_file = f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv"
df = pd.read_csv(trace_file, skipinitialspace=True)

plot_addr_hotness_timeseries(df, benchmark, time_interval=10000, top_n=10)


✅ Timeseries saved: ./addr-hotness/syr2k/syr2k_addr_access_timeseries.png
